In [14]:
import os, time, csv, requests
from datetime import datetime, timedelta
from urllib.parse import urlparse, parse_qs
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

In [16]:
# ------------------- 설정 -------------------
OUTPUT_FOLDER = "dcinside_out"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

AION2_START_PAGE = 25899
AION2_PAGE_COUNT = 100
MAX_PAGES_LOSTARK = 50  # 임시 최대 페이지, 실제 전체 수집 시 필요시 조정

CSV_AION2 = os.path.join(OUTPUT_FOLDER, "aion2_posts.csv")
CSV_LOSTARKM = os.path.join(OUTPUT_FOLDER, "lostarkm_posts.csv")

HEADERS = {"User-Agent": "Mozilla/5.0"}
keys = ["title","created_dt","view_count","like_count","selftext","link"]

# ------------------- Selenium 초기화 -------------------
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=chrome_options)

# ------------------- 함수 -------------------
def parse_date_full(date_str):
    try:
        return datetime.strptime(date_str, "%Y.%m.%d %H:%M")
    except:
        return None

def get_comments(gallery_id, post_link):
    parsed = urlparse(post_link)
    post_no = parse_qs(parsed.query).get("no", [None])[0]
    comments = []
    if post_no:
        url = f"https://gall.dcinside.com/board/comment_json/?id={gallery_id}&no={post_no}&page=1&listnum=1000"
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                for item in res.json().get("data", []):
                    comments.append(item.get("content","").replace("\n"," "))
        except:
            pass
    return comments

def scroll_page():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

def crawl_mgallery(gallery_id, start_page=1, max_pages=50, csv_file=None):
    total_posts = 0
    with open(csv_file,"w",encoding="utf-8-sig",newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for page in range(start_page, start_page + max_pages):
            try:
                url = f"https://gall.dcinside.com/mgallery/board/lists/?id={gallery_id}&page={page}"
                driver.get(url)
                time.sleep(2)
                scroll_page()
                soup = BeautifulSoup(driver.page_source, "html.parser")
                rows = soup.select("tr.ub-content")
                if not rows:
                    print(f"[{gallery_id}] 페이지 {page} → 게시글 없음 → 종료")
                    break

                for tr in rows:
                    try:
                        a_tag = tr.select_one("td.gall_tit > a")
                        if not a_tag: continue
                        href = a_tag.get("href","")
                        if "javascript:" in href: continue

                        title = a_tag.get_text(strip=True)
                        date_tag = tr.select_one("span.gall_date")
                        date_raw = date_tag.get_text(strip=True) if date_tag else ""
                        dt = parse_date_full(date_raw)
                        created_dt = dt.strftime("%Y-%m-%d %H:%M:00") if dt else ""

                        like_tag = tr.select_one("td.gall_recommend > em")
                        like_count = int(like_tag.get_text(strip=True)) if like_tag and like_tag.get_text(strip=True).isdigit() else 0

                        view_tag = tr.select_one("td.gall_count")
                        view_count = int(view_tag.get_text(strip=True).replace(",","")) if view_tag and view_tag.get_text(strip=True).replace(",","").isdigit() else 0

                        post_link = "https://gall.dcinside.com" + href
                        comments = get_comments(gallery_id, post_link)

                        total_posts += 1
                        if comments:
                            for c in comments:
                                writer.writerow({
                                    "title": title,
                                    "created_dt": created_dt,
                                    "view_count": view_count,
                                    "like_count": like_count,
                                    "selftext": c,
                                    "link": post_link
                                })
                        else:
                            writer.writerow({
                                "title": title,
                                "created_dt": created_dt,
                                "view_count": view_count,
                                "like_count": like_count,
                                "selftext": "",
                                "link": post_link
                            })
                    except Exception as e:
                        print(f"[{gallery_id}] 게시글 오류:", e)

                progress = (page-start_page+1)/max_pages*100
                print(f"[{gallery_id}] 페이지 {page}, 총 게시글 처리: {total_posts}, 진행률: {progress:.1f}%")
            except Exception as e:
                print(f"[{gallery_id}] 페이지 오류:", e)
    print(f"[{gallery_id}] CSV 저장 완료: {csv_file}, 총 게시글 수: {total_posts}")

In [17]:
# 실행
# 1. 로스트아크M 전체 수집
crawl_mgallery("lostarkm", start_page=1, max_pages=MAX_PAGES_LOSTARK, csv_file=CSV_LOSTARKM)

# 2. 아이온2 25899페이지부터 100페이지 수집
crawl_mgallery("aion2", start_page=AION2_START_PAGE, max_pages=AION2_PAGE_COUNT, csv_file=CSV_AION2)

driver.quit()

[lostarkm] 페이지 1, 총 게시글 처리: 48, 진행률: 2.0%
[lostarkm] 페이지 2, 총 게시글 처리: 99, 진행률: 4.0%
[lostarkm] 페이지 3, 총 게시글 처리: 150, 진행률: 6.0%
[lostarkm] 페이지 4, 총 게시글 처리: 201, 진행률: 8.0%
[lostarkm] 페이지 5, 총 게시글 처리: 252, 진행률: 10.0%
[lostarkm] 페이지 6, 총 게시글 처리: 303, 진행률: 12.0%
[lostarkm] 페이지 7, 총 게시글 처리: 354, 진행률: 14.0%
[lostarkm] 페이지 8, 총 게시글 처리: 405, 진행률: 16.0%
[lostarkm] 페이지 9, 총 게시글 처리: 456, 진행률: 18.0%
[lostarkm] 페이지 10, 총 게시글 처리: 507, 진행률: 20.0%
[lostarkm] 페이지 11, 총 게시글 처리: 558, 진행률: 22.0%
[lostarkm] 페이지 12, 총 게시글 처리: 609, 진행률: 24.0%
[lostarkm] 페이지 13, 총 게시글 처리: 660, 진행률: 26.0%
[lostarkm] 페이지 14, 총 게시글 처리: 711, 진행률: 28.0%
[lostarkm] 페이지 15, 총 게시글 처리: 762, 진행률: 30.0%
[lostarkm] 페이지 16, 총 게시글 처리: 813, 진행률: 32.0%
[lostarkm] 페이지 17, 총 게시글 처리: 864, 진행률: 34.0%
[lostarkm] 페이지 18, 총 게시글 처리: 915, 진행률: 36.0%
[lostarkm] 페이지 19, 총 게시글 처리: 966, 진행률: 38.0%
[lostarkm] 페이지 20, 총 게시글 처리: 1017, 진행률: 40.0%
[lostarkm] 페이지 21, 총 게시글 처리: 1068, 진행률: 42.0%
[lostarkm] 페이지 22, 총 게시글 처리: 1119, 진행률: 44.0%
[lostarkm] 페이지 23, 총 게

In [20]:
import pandas as pd
import os

OUTPUT_FOLDER = "dcinside_out"

# CSV 경로
aion2_csv = os.path.join(OUTPUT_FOLDER, "aion2_posts.csv")
lostarkm_csv = os.path.join(OUTPUT_FOLDER, "lostarkm_posts.csv")

# 불러오기
df_aion2 = pd.read_csv(aion2_csv)
df_lostarkm = pd.read_csv(lostarkm_csv)

# source 컬럼 추가
df_aion2["source"] = "dcinside_aion2"
df_lostarkm["source"] = "dcinside_lostarkm"

# region 컬럼 추가 (공통 "국내")
df_aion2["region"] = "국내"
df_lostarkm["region"] = "국내"

# 합치기
df_all = pd.concat([df_aion2, df_lostarkm], ignore_index=True)

# 결측값 처리 (selftext 비어있으면 빈 문자열)
df_all["selftext"] = df_all["selftext"].fillna("")

# 중복 제거 (제목+링크+댓글 기준)
df_all.drop_duplicates(subset=["title","link","selftext"], inplace=True)

print(f"총 게시글/댓글 수: {len(df_all)}")
print("샘플 데이터:")
print(df_all.head())

총 게시글/댓글 수: 6896
샘플 데이터:
                             title  created_dt  view_count  like_count  \
0  워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드         NaN           0           0   
1              그냥 대가리박고 십부장템 너프하죠?         NaN         183           0   
2                 마도성 pve좋다 좋다 하는데         NaN         246           0   
3               와 호법은 진짜 초월고단계지옥이네         NaN         393           0   
4                       배럭 막았음 좋겠네         NaN         208           0   

  selftext                                               link          source  \
0           https://gall.dcinside.comhttps://addc.dcinside...  dcinside_aion2   
1           https://gall.dcinside.com/mgallery/board/view/...  dcinside_aion2   
2           https://gall.dcinside.com/mgallery/board/view/...  dcinside_aion2   
3           https://gall.dcinside.com/mgallery/board/view/...  dcinside_aion2   
4           https://gall.dcinside.com/mgallery/board/view/...  dcinside_aion2   

  region  
0     국내  
1     국내  
2     국내  

In [21]:
df_all

,title,created_dt,view_count,like_count,selftext,link,source,region
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,NaN,0,0,,https://gall.dcinside.comhttps://addc.dcinside...,dcinside_aion2,국내
1,그냥 대가리박고 십부장템 너프하죠?,NaN,183,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_aion2,국내
2,마도성 pve좋다 좋다 하는데,NaN,246,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_aion2,국내
3,와 호법은 진짜 초월고단계지옥이네,NaN,393,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_aion2,국내
4,배럭 막았음 좋겠네,NaN,208,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_aion2,국내
...,...,...,...,...,...,...,...,...
7028,응애,NaN,82,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_lostarkm,국내
7029,뿌직,NaN,93,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_lostarkm,국내
7030,뭔데여기,NaN,256,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_lostarkm,국내
7031,갤 어케뚫었냐 씨1발 알바새끼 내가 만들려고할땐,NaN,1300,0,,https://gall.dcinside.com/mgallery/board/view/...,dcinside_lostarkm,국내


In [22]:
# 필요한 컬럼 선택
df_selected = df_all[["title","source","region"]].copy()

# title 컬럼 이름 변경 → selftext
df_selected.rename(columns={"title":"selftext"}, inplace=True)

In [23]:
df_selected

,selftext,source,region
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내
4,배럭 막았음 좋겠네,dcinside_aion2,국내
...,...,...,...
7028,응애,dcinside_lostarkm,국내
7029,뿌직,dcinside_lostarkm,국내
7030,뭔데여기,dcinside_lostarkm,국내
7031,갤 어케뚫었냐 씨1발 알바새끼 내가 만들려고할땐,dcinside_lostarkm,국내


In [24]:
# 중복 확인 및 삭제 (selftext+source+region 기준)
duplicates_mask = df_selected.duplicated(subset=["selftext","source","region"], keep="first")
duplicates_count = duplicates_mask.sum()
print(f"중복 행 수: {duplicates_count}")

중복 행 수: 186


In [25]:
df_selected = df_selected[~duplicates_mask]
print(f"중복 제거 후 데이터 수: {len(df_selected)}")

중복 제거 후 데이터 수: 6710


In [27]:
# 링크(http/https) 포함 행 조회 및 삭제
link_mask = df_selected["selftext"].str.contains("http|https", case=False, na=False)
link_count = link_mask.sum()
print(f"링크 포함 행 수: {link_count}")

링크 포함 행 수: 0


In [28]:
# 최종 CSV 저장
cleaned_csv = os.path.join(OUTPUT_FOLDER, "dcinside_all_posts.csv")
df_selected.to_csv(cleaned_csv, index=False, encoding="utf-8-sig")

print(f"전처리 완료 CSV 저장: {cleaned_csv}")

전처리 완료 CSV 저장: dcinside_out\dcinside_all_posts.csv


In [32]:
import pandas as pd
import os
import glob

# 3_comment_original 폴더 경로
INPUT_FOLDER = r"F:\porti\2026\Control_Defines_Difficulty\Control_Defines_Difficulty\3_comment_original"
OUTPUT_FOLDER = "ALL_out"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 폴더 안 CSV 목록 가져오기
csv_files = glob.glob(os.path.join(INPUT_FOLDER, "*.csv"))
print(f"합칠 CSV 파일 수: {len(csv_files)}")
print(csv_files)

합칠 CSV 파일 수: 4
['F:\\porti\\2026\\Control_Defines_Difficulty\\Control_Defines_Difficulty\\3_comment_original\\dcinside_all_posts.csv', 'F:\\porti\\2026\\Control_Defines_Difficulty\\Control_Defines_Difficulty\\3_comment_original\\inven_all_posts.csv', 'F:\\porti\\2026\\Control_Defines_Difficulty\\Control_Defines_Difficulty\\3_comment_original\\reddit_all_posts_translated.csv', 'F:\\porti\\2026\\Control_Defines_Difficulty\\Control_Defines_Difficulty\\3_comment_original\\youtube_public_playlists_full_last2.csv']


In [33]:
# 공통 칼럼 선택하며 불러오기
dfs = []
for f in csv_files:
    df = pd.read_csv(f, usecols=["selftext","source","region"])
    dfs.append(df)

# 합치기
df_all_comments = pd.concat(dfs, ignore_index=True)

In [34]:
df_all_comments

,selftext,source,region
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내
4,배럭 막았음 좋겠네,dcinside_aion2,국내
...,...,...,...
16564,ㅄ같은 마사지를 보았습니다,youtube,국내
16565,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내
16566,lets gooooooo plz don't make same mistakes yo...,youtube,국내
16567,클베 언제인지 아시나요?,youtube,국내


In [35]:
# 결측값 처리
df_all_comments["selftext"] = df_all_comments["selftext"].fillna("")
df_all_comments["source"] = df_all_comments["source"].fillna("")
df_all_comments["region"] = df_all_comments["region"].fillna("")

# 중복 제거 (selftext+source+region 기준)
df_all_comments.drop_duplicates(subset=["selftext","source","region"], inplace=True)

print(f"총 행 수: {len(df_all_comments)}")
print(df_all_comments.head())

총 행 수: 14933
                          selftext          source region
0  워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드  dcinside_aion2     국내
1              그냥 대가리박고 십부장템 너프하죠?  dcinside_aion2     국내
2                 마도성 pve좋다 좋다 하는데  dcinside_aion2     국내
3               와 호법은 진짜 초월고단계지옥이네  dcinside_aion2     국내
4                       배럭 막았음 좋겠네  dcinside_aion2     국내


In [36]:
# 최종 CSV 저장
merged_comments_csv = os.path.join(OUTPUT_FOLDER, "all_merged_comments.csv")
df_all_comments.to_csv(merged_comments_csv, index=False, encoding="utf-8-sig")

In [37]:
df_all_comments

,selftext,source,region
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내
4,배럭 막았음 좋겠네,dcinside_aion2,국내
...,...,...,...
16564,ㅄ같은 마사지를 보았습니다,youtube,국내
16565,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내
16566,lets gooooooo plz don't make same mistakes yo...,youtube,국내
16567,클베 언제인지 아시나요?,youtube,국내
